# 第 5 週 實作｜積分入門（Colab notebook）

**目標**：把理論課的積分概念**親手跑出來、畫出來**——
1. `riemann_sum(f,a,b,n)`：看黎曼和隨 `n` 增大逼近 $\int_0^1 x^2\,dx = 1/3$
2. 比較 **左 / 中點 / 梯形 / Simpson** 的誤差如何隨 `n` 縮小（error vs n，連到演算法的成長率）
3. `numpy.trapezoid` / `scipy.integrate.quad`：現成的數值積分
4. **蒙地卡羅**估 $\pi$
5. 數值驗證**微積分基本定理**：黎曼和的 $\int_a^b f$ 對上 $F(b)-F(a)$

**用法**：上傳到 [Google Colab](https://colab.research.google.com/) 或本機 Jupyter，由上往下逐格執行。
標「`# TODO 學生練習`」的格子留給你填。

> 圖表標籤用英文/數學符號以避免中文變豆腐字；中文都放在說明格。

In [ ]:
# === 環境設定(先跑這格)===
import math
import numpy as np
import matplotlib.pyplot as plt
from scipy import integrate as si

plt.rcParams['figure.figsize'] = (7, 4.5)
plt.rcParams['axes.grid'] = True
print("環境就緒, numpy", np.__version__)

### (選用)讓圖表顯示中文

預設圖表用英文標籤,避免中文變「豆腐字」。若你在 Colab 想要中文座標/標題,
把下一格的註解取消再執行(只需一次),之後的圖就能顯示中文。

In [ ]:
# 想要中文圖標時,取消以下註解執行(Colab 適用;本機 Jupyter 需自備 CJK 字型)
# !apt-get -qq install fonts-noto-cjk > /dev/null
# import matplotlib
# matplotlib.rcParams['font.sans-serif'] = ['Noto Sans CJK TC']
# matplotlib.rcParams['axes.unicode_minus'] = False
print("預設英文標籤;要中文請見上一格說明")

## Lab 1｜黎曼和逼近定積分

`riemann_sum(f, a, b, n)` 把 `[a,b]` 切成 `n` 條、用取樣點的高度堆矩形。
理論課手算了 $\int_0^1 x^2\,dx = 1/3$;這裡讓 `n` 一路長大,看數值一步步爬向 `0.3333...`。

In [ ]:
def riemann_sum(f, a, b, n, kind="right"):
    """kind: 'left' / 'right' / 'mid'。回傳 sum f(x_i) * dx。"""
    dx = (b - a) / n
    i = np.arange(n)
    if kind == "left":
        xs = a + i * dx
    elif kind == "right":
        xs = a + (i + 1) * dx
    else:  # midpoint
        xs = a + (i + 0.5) * dx
    return np.sum(f(xs)) * dx

f = lambda x: x**2
exact = 1/3
print(f"exact int_0^1 x^2 dx = {exact:.10f}\n")
print(" n      right-sum      error")
for n in [1, 2, 4, 10, 100, 1000, 10000]:
    approx = riemann_sum(f, 0, 1, n, "right")
    print(f"{n:6d}   {approx:.8f}   {abs(approx-exact):.2e}")

In [ ]:
# 視覺化:n=10 的右黎曼和矩形,vs 真正的曲線
n = 10
dx = 1 / n
xr = (np.arange(n) + 1) * dx          # 右端點
plt.bar(np.arange(n) * dx, f(xr), width=dx, align='edge',
        alpha=0.35, edgecolor='C0', label=f'right rectangles (n={n})')
xx = np.linspace(0, 1, 400)
plt.plot(xx, f(xx), 'C3', lw=2, label='y = x^2')
plt.title('Riemann sum overestimates area under x^2 (increasing f)')
plt.legend(); plt.show()

## Lab 2｜左 / 中點 / 梯形 / Simpson 的誤差如何縮小

不同規則收斂速度差很多,這正是**演算法成長率**的味道:
- **左端點**:誤差 $\sim h^{1}$(每加倍 `n`,誤差減半)
- **中點、梯形**:誤差 $\sim h^{2}$(誤差變 1/4)
- **Simpson**:誤差 $\sim h^{4}$(誤差變 1/16)——花一樣多的點,精準度天差地遠。

用 $f(x)=e^x$ 在 `[0,1]`(真值 `e - 1`),把 error 對 `n` 畫在 log-log 上,斜率就是成長率。

In [ ]:
def trapezoid_rule(f, a, b, n):
    x = np.linspace(a, b, n + 1)
    y = f(x)
    h = (b - a) / n
    return h * (0.5 * y[0] + y[1:-1].sum() + 0.5 * y[-1])

def simpson_rule(f, a, b, n):
    if n % 2 == 1:      # Simpson 需要偶數段
        n += 1
    x = np.linspace(a, b, n + 1)
    y = f(x)
    h = (b - a) / n
    return h / 3 * (y[0] + 4 * y[1:-1:2].sum() + 2 * y[2:-2:2].sum() + y[-1])

g = lambda x: np.exp(x)
exact = np.e - 1
ns = np.array([2, 4, 8, 16, 32, 64, 128, 256])

err = {k: [] for k in ["left", "mid", "trapezoid", "Simpson"]}
for n in ns:
    err["left"].append(abs(riemann_sum(g, 0, 1, n, "left") - exact))
    err["mid"].append(abs(riemann_sum(g, 0, 1, n, "mid") - exact))
    err["trapezoid"].append(abs(trapezoid_rule(g, 0, 1, n) - exact))
    err["Simpson"].append(abs(simpson_rule(g, 0, 1, n) - exact))

for k, v in err.items():
    plt.loglog(ns, v, 'o-', label=k)
plt.xlabel('n (number of strips)'); plt.ylabel('absolute error')
plt.title('Error vs n:  steeper slope = faster convergence')
plt.legend(); plt.show()

print("每次 n 加倍,誤差大約乘上的倍率(越小越快):")
for k, v in err.items():
    ratios = [v[i+1] / v[i] for i in range(len(v) - 1) if v[i] > 0]
    print(f"  {k:10s} ~ x{np.mean(ratios):.3f}   (1/2->O(h), 1/4->O(h^2), 1/16->O(h^4))")

## Lab 3｜現成的數值積分:`numpy.trapezoid` 與 `scipy.integrate.quad`

自己寫過一輪後,平常就用函式庫。`np.trapezoid`(舊名 `np.trapz`)吃取樣點做梯形法;
`scipy.integrate.quad` 是自動選步長的高精度積分器,還回報誤差估計。

In [ ]:
# numpy 梯形法(新版叫 trapezoid;舊版叫 trapz)
trapz = getattr(np, "trapezoid", np.trapz)
x = np.linspace(0, 1, 1001)
print("np.trapezoid  int_0^1 x^2 dx  =", trapz(x**2, x))

# scipy.quad:回傳 (積分值, 誤差估計)
val, errest = si.quad(lambda t: t**2, 0, 1)
print("scipy.quad    int_0^1 x^2 dx  =", val, " (誤差估計 ~", f"{errest:.1e})")

# 換個不好手算的:int_0^1 e^{-x^2} dx(高斯型,沒有初等反導數)
val2, _ = si.quad(lambda t: np.exp(-t**2), 0, 1)
print("scipy.quad    int_0^1 e^{-x^2} dx =", val2)

## Lab 4｜蒙地卡羅估 $\pi$

在單位正方形 `[0,1]x[0,1]` 隨機灑點,落在四分之一圓 `x^2 + y^2 <= 1` 內的比例
$\approx \dfrac{\text{圓面積}/4}{\text{正方形面積}} = \dfrac{\pi/4}{1}$,所以 $\pi \approx 4 \times$ 命中率。
這其實就是「用隨機取樣估面積(積分)」——一種機率版的黎曼和。

In [ ]:
rng = np.random.default_rng(0)
N = 200_000
pts = rng.random((N, 2))
inside = pts[:, 0]**2 + pts[:, 1]**2 <= 1.0
pi_est = 4 * inside.mean()
print(f"N = {N} 個隨機點,估計 pi ~ {pi_est:.5f}   (真值 {math.pi:.5f})")

# 畫出來(只畫前 3000 點免得太擠)
s = pts[:3000]
ins = s[:, 0]**2 + s[:, 1]**2 <= 1
plt.figure(figsize=(5, 5))
plt.scatter(s[ins, 0],  s[ins, 1],  s=3, c='C0', label='inside')
plt.scatter(s[~ins, 0], s[~ins, 1], s=3, c='C3', label='outside')
th = np.linspace(0, np.pi/2, 200)
plt.plot(np.cos(th), np.sin(th), 'k', lw=1.5)
plt.gca().set_aspect('equal'); plt.title(f'Monte Carlo pi ~ {pi_est:.4f}')
plt.legend(); plt.show()

In [ ]:
# TODO 學生練習:誤差 ~ 1/sqrt(N)。把 N 從 10^2 掃到 10^6,畫 |pi_est - pi| vs N(log-log)
# Ns = np.logspace(2, 6, 12).astype(int)
# errs = []
# for n in Ns:
#     p = rng.random((n, 2))
#     errs.append(abs(4 * np.mean(p[:,0]**2 + p[:,1]**2 <= 1) - math.pi))
# plt.loglog(Ns, errs, 'o-'); plt.loglog(Ns, 1/np.sqrt(Ns), '--', label='1/sqrt(N)')
# plt.legend(); plt.xlabel('N'); plt.ylabel('error'); plt.show()

## Lab 5｜數值驗證微積分基本定理(FTC)

**FTC 第二部分**:$\int_a^b f(x)\,dx = F(b) - F(a)$,其中 $F' = f$。
左邊用**黎曼和**硬算(數值),右邊用**反導數**代端點(解析),兩邊應該幾乎相等——
這就是「積分 = 反微分」在數字上的證據。

In [ ]:
# f = 3x^2 + 1,反導數 F = x^3 + x
f = lambda x: 3 * x**2 + 1
F = lambda x: x**3 + x
a, b = 1.0, 2.0

lhs = riemann_sum(f, a, b, 200_000, "mid")   # 數值:黎曼和
rhs = F(b) - F(a)                            # 解析:F(b) - F(a)
print(f"黎曼和   int_a^b f dx = {lhs:.8f}")
print(f"FTC      F(b) - F(a)  = {rhs:.8f}")
print(f"差距                  = {abs(lhs - rhs):.2e}   -> 幾乎為 0,FTC 成立")

In [ ]:
# 也驗 FTC 第一部分:g(x)=int_0^x f 的導數應該等於 f(x)
# 取 f(t)=sin(t),則 g(x)=int_0^x sin t dt = 1 - cos x,g'(x) 應為 sin x
xs = np.linspace(0.2, 3.0, 12)
g  = lambda x: si.quad(lambda t: np.sin(t), 0, x)[0]   # 面積函數(數值)
h  = 1e-6
gprime = np.array([(g(x + h) - g(x - h)) / (2 * h) for x in xs])  # 數值微分
print(" x      g'(x)~     sin(x)     diff")
for x, gp in zip(xs, gprime):
    print(f"{x:4.2f}  {gp:8.5f}  {math.sin(x):8.5f}  {abs(gp-math.sin(x)):.1e}")
print("\ng'(x) ~ sin(x) -> 先積分再微分,轉一圈回到原函數(FTC1)")

## 收尾 · 與筆試的連結

| 這格實作 | 對應觀念 |
|---|---|
| Lab 1 `riemann_sum` 逼近 1/3 | 黎曼和、定積分定義(觀念 3、4) |
| Lab 2 左/中/梯形/Simpson 誤差 | 黎曼和取樣、數值積分收斂(觀念 3;連演算法成長率) |
| Lab 3 `trapezoid` / `quad` | 定積分計算(觀念 4、7) |
| Lab 4 蒙地卡羅估 pi | 積分 = 面積、隨機取樣(觀念 4) |
| Lab 5 黎曼和 vs `F(b)-F(a)` | 微積分基本定理 FTC 1 & 2(觀念 6、7) |

**核心主線**:differentiation 與 integration **互為逆運算**(呼應第 2 週)。
Lab 5 就是這條主線的數值證據:先積分再微分會轉回原函數。

### 進階徽章(選做)
1. 完成 Lab 4 的 TODO:驗證蒙地卡羅誤差 $\sim 1/\sqrt{N}$(比黎曼和慢,但不怕高維度)。
2. 把 Lab 2 的被積函數換成 $\frac{1}{1+x^2}$(真值 $\arctan 1=\pi/4$),看四種規則的斜率是否一致。
3. 用 `sympy`:`sp.integrate(3*x**2+1, (x, 1, 2))` 對答案,再和 Lab 5 的數值比一比。